# Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, trim
from pyspark.sql.types import StringType
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
RENAME_MAP = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_key",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales",
    "sls_quantity": "quantity",
    "sls_price": "price"
}

# Reading from Bronze layer

In [0]:
df = spark.table("data_lakehouse_project.bronze.crm_sales_details")

# Data Transformation

- TRIM the strings
- UPPER the strings
- Managed "null"
- date type
- Rename Columns

In [0]:
df.display()

## Triming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## Upper STRING columns

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, F.upper(col(field.name)))

## Nulls

"sls_price" and "sls_sales" seems to be the same values on each, and it was the only 2 columns with NULL, so when 1 is NULL I take the value from the other.

In [0]:
df = df.withColumn(
    "sls_sales",
    F.when(F.col("sls_sales").isNull(), F.col("sls_price")).otherwise(F.col("sls_sales"))
).withColumn(
    "sls_price",
    F.when(F.col("sls_price").isNull(), F.col("sls_sales")).otherwise(F.col("sls_price"))
)

"order_date" have some NULL, decide to replace by the max difference in days from the "sls_ship_dt"

## Date

Add a dot "." to have the date correct from "20102112" to become "2010.12.21" and the CAST to a date type format

In [0]:
for col_name in ["sls_order_dt", "sls_ship_dt", "sls_due_dt"]:
    df = df.withColumn(
        col_name,
        F.when(
            (F.col(col_name).isNull()) | (F.col(col_name) < 10000000),
            F.lit(None).cast("date")
        ).otherwise(
            F.to_date(
                F.concat_ws(
                    ".",
                    F.substring(F.col(col_name).cast("string"), 1, 4),
                    F.substring(F.col(col_name).cast("string"), 5, 2),
                    F.substring(F.col(col_name).cast("string"), 7, 2)
                ),
                "yyyy.MM.dd"
            )
        )
    )

In [0]:
max_diff = df.select(
    F.max(F.datediff(F.col("sls_ship_dt"), F.col("sls_order_dt"))).alias("max_diff")
).collect()[0]["max_diff"]

df = df.withColumn(
    "sls_order_dt",
    F.when(
        F.col("sls_order_dt").isNull(),
        F.date_add(F.col("sls_ship_dt"), -max_diff)
    ).otherwise(F.col("sls_order_dt"))
)

## Rename Columns

In [0]:
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

# Write into Silver Table

In [0]:
(
    df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("data_lakehouse_project.silver.crm_sales")
)

# Check queries

In [0]:
%sql
SELECT 
    *
FROM data_lakehouse_project.silver.crm_sales 